In [1]:
import re
import time
import pandas as pd

In [2]:
# load the dataset
try:
    df = pd.read_csv("IMDB Dataset.csv")
except FileNotFoundError as error:
    raise FileNotFoundError(f"File not found! Check the file path") from error
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# find the size of positive and negative reviews
def number_reviews(df:pd.DataFrame) -> tuple[float]:
    """
    This function finds the length of positive and negative review
    Args:
        df:the IMDB Dataset.csv dataset file
    return (positive_reviews_length, negative_reviews_length)
    """
    positive_reviews_length = df[df["sentiment"] == "positive"].shape[0]
    negative_reviews_length = df[df["sentiment"] == "negative"].shape[0]
    return positive_reviews_length, negative_reviews_length

number_reviews(df=df)

(25000, 25000)

In [4]:
# choosing the keyword
positive_words = ["wonderful","masterpiece", "brilliant", "love"]
negative_words = ["boring", "dreadful", "disgraceful", "painful"]

### In this implementation, we chose to find the conditional positive probability of all the two lists chosen.

### We start with the positive_words list then We complete with the negative_words list

In [5]:
def contain_keyword(text:str, keyword:str) -> bool:
    """
    This is an helper function to handle reviews
    Args:
        text:str (The part of the review to filter to find the word)
        keyword:str (The word to look for)
    return True if found overwise False
    """
    return bool(re.search(rf"\b{re.escape(keyword)}\b", text, re.IGNORECASE))


In [6]:
def count_reviews_with_keyword(df:pd.DataFrame, keyword:str, sentiment_type:str) -> int:
    """
    Helper function to find the number of times the keyword appears in reviews
    of the given sentiment_type (works for either "positive" or "negative").
    Args:
        df: the IMDB dataset for this problem
        keyword:str (the word to look for)
        sentiment_type: Either positive or negative
    """
    sentiment_type = sentiment_type.lower()
    count = 0
    for reviews in df[df["sentiment"] == sentiment_type]["review"]:
        if contain_keyword(text=reviews, keyword=keyword):
            count += 1

    return count

In [7]:
def probability_keyword_given_sentiment(keyword_count:int, sentiment_type:str) -> float:
    """
    This helper function finds the likelihood P(keyword | sentiment_type) of 
    finding a keyword in a review of the given sentiment_type 
    (works for either "positive" or "negative")
    Args:
        keyword_count: int = The number of time the word is found in a particular review (positive or negative reviews).
        This value is retrieved from the count_reviews_with_keyword function.
        sentiment_type:str = type of review (either positive or negative)
    return probability

    """
    sentiment_type = sentiment_type.lower()
    sentiment_index = 0 if sentiment_type == "positive" else 1
    probability = (keyword_count/number_reviews(df=df)[sentiment_index])
    return probability

In [8]:
# Implementation example
probability_keyword_given_sentiment(
    keyword_count=count_reviews_with_keyword(df=df, keyword=positive_words[0], sentiment_type="positive"),
    sentiment_type="positive"
)

0.09032

In [9]:
def probability_complement(probability:float) -> float:
    """
    Helper function to get the complement of a probability, P(not event) = 1 - P(event)
    (works for either a positive- or negative-conditioned probability)
    Args:
        probability:float = the probability obtained by running probability_keyword_given_sentiment function
    return complement probability
    """
    return 1 - probability

In [10]:
# Implementation example for complement probability
probability_complement(
    probability=probability_keyword_given_sentiment(
        keyword_count=count_reviews_with_keyword(df=df, keyword=positive_words[0], sentiment_type="positive"),
        sentiment_type="positive"
    )
)

0.90968

In [11]:
def probability_keyword_marginal(df:pd.DataFrame, keyword:str) -> float:
    """
    Helper function to get P(keyword): the marginal probability
    of the keyword appearing anywhere in the dataset, regardless of sentiment
    Args:
        df:the dataset provided
        keyword:str = the keyword to look for
    return marginal_probability
    """
    count = 0
    for reviews in df["review"]:
        if contain_keyword(text=reviews, keyword=keyword):
            count += 1
    probability = count/len(df)
    return probability

In [12]:
positive_review = (number_reviews(df=df)[0]/len(df))*100
negative_review = (number_reviews(df=df)[1]/len(df))*100
print(f"probability of finding a positive review in the dataset is: {positive_review:.2f}%")
print(f"probability of finding a negative review in the dataset is: {negative_review:.2f}%")

probability of finding a positive review in the dataset is: 50.00%
probability of finding a negative review in the dataset is: 50.00%


In [13]:
def bayesian_probability(df: pd.DataFrame, keyword: str, sentiment_type: str) -> float:
    """
    Computes P(sentiment_type | keyword) via Bayes' theorem. This function reuses
    all helpers functions, work for either 'positive' or 'negative'
    Args:
        df = The provided dataset
        keyword:str = The word to look for
        sentiment_type = The review type (positive or negative)
    return conditional_probability
    """
    sentiment_index = 0 if sentiment_type == "positive" else 1

    keyword_count = count_reviews_with_keyword(df=df, keyword=keyword, sentiment_type=sentiment_type)

    p_keyword_given_sentiment = probability_keyword_given_sentiment(keyword_count=keyword_count, sentiment_type=sentiment_type)
    p_sentiment = number_reviews(df=df)[sentiment_index] / len(df)
    p_keyword = probability_keyword_marginal(df=df, keyword=keyword)

    try:
        conditional_probability = (p_keyword_given_sentiment * p_sentiment) / p_keyword
    except ZeroDivisionError:
        conditional_probability = 0

    return {
        "keyword": keyword,
        "prior_probability in %":p_sentiment,
        "likelihood in %":p_keyword_given_sentiment,
        "marginal_probability in %":p_keyword,
        "posterior in %":conditional_probability
    }


In [14]:
print("Display all posterior probabilities ")
start = time.time()
probability_table_df = []
for keyword in positive_words + negative_words:
    probability = bayesian_probability(df=df, keyword=keyword, sentiment_type="positive")
    probability_table_df.append(probability)
    print(f"P(positive | '{keyword}') = {probability["posterior in %"]:.3f} ({probability["posterior in %"]*100:.2f}%)")
end = time.time()
print(f"\nLoop-based computation took {end - start:.3f} seconds")

Display all posterior probabilities 
P(positive | 'wonderful') = 0.812 (81.22%)
P(positive | 'masterpiece') = 0.727 (72.74%)
P(positive | 'brilliant') = 0.760 (76.01%)
P(positive | 'love') = 0.635 (63.49%)
P(positive | 'boring') = 0.194 (19.37%)
P(positive | 'dreadful') = 0.137 (13.73%)
P(positive | 'disgraceful') = 0.108 (10.81%)
P(positive | 'painful') = 0.231 (23.11%)

Loop-based computation took 30.053 seconds


In [15]:
# run table of values
import json
#print(json.dumps(probability_table, indent=2))
percentage_table_df = pd.DataFrame(data=probability_table_df)
probability_table = percentage_table_df.set_index("keyword") * 100
probability_table.round(2)



,prior_probability in %,likelihood in %,marginal_probability in %,posterior in %
keyword,,,,
wonderful,50.0,9.03,5.56,81.22
masterpiece,50.0,3.51,2.41,72.74
brilliant,50.0,6.35,4.18,76.01
love,50.0,22.66,17.85,63.49
boring,50.0,2.36,6.10,19.37
dreadful,50.0,0.25,0.92,13.73
disgraceful,50.0,0.02,0.07,10.81
painful,50.0,0.71,1.53,23.11
